# Script 03: pegRNA oligo ordering for Golden Gate assembly

Author: Kexin Dong

Built: 2026-09-29

Takes designed pegRNAs (PEGG `prime.run` output, or PRIDICT 2.0 output) and outputs an
IDT-ready order table — replacing the three Google Sheets in the
[pegRNA GGA protocol](https://docs.google.com/document/d/1JMf-9BgKL8n8L1rh6DR5F0PcneS9fLbj9ErIh_BYh_I/edit).

## The three cloning cases

The 3' extension oligos depend on **which vector you cut** and **where the epeg motif comes
from**. Overhangs below were read directly off the SnapGene maps (Addgene 196036/196037 and
the `-tevo` derivatives), not copied from a spreadsheet:

| `backbone` | `epeg` | upstream OH | downstream OH | 3' ext top | 3' ext bottom |
|---|---|---|---|---|---|
| `UPEmS-tevo`, `UPEC-tevo` | `'backbone'` | `CACC` | `CGCG` | `gtgc`+ext | `cgcg`+rc(ext) |
| `UPEmS`, `UPEC` | `'oligo'` | `CACC` | `TTTT` | `gtgc`+ext+tevo | `aaaa`+rc(tevo)+rc(ext) |
| `UPEmS`, `UPEC` | `'none'` | `CACC` | `TTTT` | `gtgc`+ext | `aaaa`+rc(ext) |

`CGCG` is literally the first 4 nt of tevopreQ1, which is why the `-tevo` backbones join there.

```{important}
The old `tevopreQ1=True/False` flag collapsed the last two rows into one and read backwards
(`False` meant "tevo is already in the backbone", not "no tevo"). `backbone` + `epeg` are now
separate, and impossible combinations raise instead of quietly producing dead oligos — a
`cgcg` bottom oligo in plain UPEmS, or an `aaaa` one in UPEmS-tevo, gives no colonies.
```

## What this notebook adds over the spreadsheets

- **Scaffold stays out of the order sheet** — you keep phosphorylated stock. Sequences are in
  the last cell for when it runs out.
- **BsmBI scan** of protospacer, extension and every junction. A `CGTCTC`/`GAGACG` inside the
  RTT/PBS silently destroys the assembly; this raises instead.
- **Dedup**: pegRNAs sharing a protospacer share `P_T`/`P_B`. Ordered once.
- **Correct IDT scale by length** — `epeg='oligo'` extension oligos run 60–95 nt and need
  100 nm, not the 25 nm the old code hard-coded.
- **In-silico Golden Gate** against the real `.dna` map, replacing the manual SnapGene step
  the protocol calls "HIGHLY recommended".


## Setup

In [37]:
# !pip install biopython pandas openpyxl
import pandas as pd

pd.set_option("display.max_colwidth", 200)

## Config — edit this cell only

In [ ]:
# ============================== CONFIG ==============================
INPUT      = '/Users/kexindong/Documents/GitHub/early-leukemia-evolution/ch_aml_pegRNAs_pilot.csv'  # PEGG/PRIDICT csv; or assign DESIGNS below
NAME       = "CH_AML"                # oligo name prefix -> NAME_index_gene_mut_P_T
BACKBONE   = "UPEC-tevo"             # UPEmS | UPEC | UPEmS-tevo | UPEC-tevo
EPEG       = None                    # -tevo: leave None. Plain: 'oligo' or 'none'
COLUMNS    = "pegg"                  # 'pegg' (Protospacer/RTT_PBS) | 'pridict'
GENE_COL   = "Hugo_Symbol"           # gene label for the oligo name; None to omit
MUT_COL    = "HGVSp_m"               # mutation label for the oligo name; None to omit
VECTOR_MAP = '/Users/kexindong/Desktop/HSC x Order/Assembly/UPEC_tevo.dna'  # "/path/to/UPEmS_tevo.dna" -> runs in-silico GGA
OUTPUT_DIR = "/Users/kexindong/Documents/GitHub/early-leukemia-evolution/pegrna_orders"
PLATE      = False                   # True -> add Well Position for IDT plate upload
# ====================================================================

## Functions

Run these once; nothing below needs editing.

In [ ]:
from __future__ import annotations

import re
from dataclasses import dataclass, replace
from typing import Iterable, Sequence

import pandas as pd

# --- sequence primitives -----------------------------------------------------

# str.translate is the fastest pure-Python reverse complement.
_COMP = str.maketrans("ACGTUNacgtun", "TGCAANtgcaan")


def rc(seq: str) -> str:
    """Reverse complement."""
    return seq.translate(_COMP)[::-1]


# Fixed parts of the pegRNA cassette. SCAFFOLD is the sgRNA scaffold minus the
# 5' GTTTTAGAG and the 3' GTGC, which live on the flanking oligos as overhangs.
SCAFFOLD = "CTAGAAATAGCAAGTTAAAATAAGGCTAGTCCGTTATCAACTTGAAAAAGTGGCACCGAGTCG"
TEVOPREQ1 = "CGCGGTTCTATCTAGTTACGCGTTAAACCAACTAGAA"

# Golden Gate junctions, as they appear on the plasmid top strand.
OH_VECTOR_UP = "CACC"      # U6 +1 -> protospacer
OH_PROTO_SCAF = "AGAG"     # protospacer -> scaffold (top strand)
OH_SCAF_EXT = "GTGC"       # scaffold -> 3' extension (top strand)

# BsmBI / Esp3I, CGTCTC(1/5). Both strands must be scanned: the site is not
# palindromic.
_BSMBI_FWD = re.compile("CGTCTC")
_BSMBI_REV = re.compile("GAGACG")
_POLYT = re.compile("T{5,}")

In [ ]:
# --- backbones ---------------------------------------------------------------

@dataclass(frozen=True)
class Backbone:
    """A BsmBI-linearizable pegRNA vector.

    down_overhang is the 4-nt 5' top-strand overhang the vector presents
    downstream of the insert; it is what decides the 3' extension bottom oligo.
    """

    name: str
    up_overhang: str
    down_overhang: str
    has_tevopreQ1: bool

    def epeg_options(self) -> tuple[str, ...]:
        return ("backbone",) if self.has_tevopreQ1 else ("oligo", "none")


# Verified against the SnapGene maps (Addgene 196036/196037 and the -tevo derivatives).
BACKBONES: dict[str, Backbone] = {
    "UPEC": Backbone("UPEC", "CACC", "TTTT", False),
    "UPEmS": Backbone("UPEmS", "CACC", "TTTT", False),
    "UPEC-tevo": Backbone("UPEC-tevo", "CACC", "CGCG", True),
    "UPEmS-tevo": Backbone("UPEmS-tevo", "CACC", "CGCG", True),
}


def resolve_backbone(backbone: str | Backbone) -> Backbone:
    if isinstance(backbone, Backbone):
        return backbone
    key = str(backbone).strip()
    for candidate in (key, key.replace("_", "-"), key.replace("Cre", "C")):
        if candidate in BACKBONES:
            return BACKBONES[candidate]
    raise KeyError(
        f"Unknown backbone {backbone!r}. Known: {sorted(BACKBONES)}. "
        "For a new vector, build a Backbone(...) or use backbone_from_map()."
    )


def resolve_epeg(bb: Backbone, epeg: str | None) -> str:
    """Validate the epeg choice against the backbone; never guess silently."""
    allowed = bb.epeg_options()
    if epeg is None:
        if bb.has_tevopreQ1:
            return "backbone"
        raise ValueError(
            f"{bb.name} carries no tevopreQ1, so epeg= must be stated explicitly: "
            "'oligo' (3' extension oligos supply the motif) or 'none' (plain pegRNA)."
        )
    epeg = epeg.lower()
    if epeg not in allowed:
        raise ValueError(
            f"epeg={epeg!r} is impossible for {bb.name} "
            f"(tevopreQ1 in backbone: {bb.has_tevopreQ1}). Allowed: {allowed}."
        )
    return epeg


# --- column presets ----------------------------------------------------------

@dataclass(frozen=True)
class ColumnMap:
    """Where to find protospacer / 3' extension / design id in the input table.

    gene and mut are only used to label the oligos; None leaves that part of the
    name out.
    """

    protospacer: str
    extension: str
    design_id: str
    gene: str | None = None
    mut: str | None = None
    strip_prefix: int = 0  # PRIDICT ships oligo columns already carrying overhangs


COLUMN_PRESETS: dict[str, ColumnMap] = {
    "pegg": ColumnMap("Protospacer", "RTT_PBS", "pegRNA_id",
                      gene="Hugo_Symbol", mut="HGVSp_m"),
    "pridict": ColumnMap(
        "Protospacer-Oligo-FW", "Extension-Oligo-FW", "rank", strip_prefix=4
    ),
}

In [41]:
# --- oligo construction ------------------------------------------------------

@dataclass(frozen=True)
class Duplex:
    """An annealed oligo pair, named by its role in the assembly."""

    role: str
    top: str
    bot: str

    @property
    def left(self) -> str:
        """4-nt 5' overhang carried on the top strand."""
        return self.top[:4]

    @property
    def right(self) -> str:
        """4-nt 5' overhang carried on the bottom strand."""
        return self.bot[:4]

    @property
    def top_contribution(self) -> str:
        """What this duplex adds to the final top strand (overhang + duplexed core)."""
        return self.top

    def validate(self) -> None:
        if len(self.top) != len(self.bot):
            raise ValueError(
                f"{self.role}: top ({len(self.top)} nt) and bottom ({len(self.bot)} nt) "
                "must be equal length for 4-nt overhangs on both ends."
            )
        core = self.top[4:]
        if not rc(self.bot).startswith(core):
            raise ValueError(f"{self.role}: oligos do not anneal — core is not complementary.")


def normalize_protospacer(proto: str, g_start: str = "prepend") -> tuple[str, str]:
    """Force a G at position 1 for the U6 promoter. Returns (spacer, note)."""
    p = proto.strip().upper()
    if p.startswith("G"):
        return p, ""
    if g_start == "prepend":
        return "G" + p, f"G prepended ({len(p)}->{len(p) + 1} nt spacer)"
    if g_start == "replace":
        return "G" + p[1:], f"5' {p[0]}->G substituted"
    raise ValueError(f"Protospacer {p} does not start with G and g_start='require'.")


def protospacer_duplex(spacer: str) -> Duplex:
    # GTTTT/AAAAC is the 5' end of the sgRNA scaffold, split across the junction.
    return Duplex("protospacer", "cacc" + spacer + "gtttt", "ctctaaaac" + rc(spacer))


def scaffold_duplex() -> Duplex:
    return Duplex("scaffold", "agag" + SCAFFOLD, "gcac" + rc(SCAFFOLD))


def extension_duplex(ext: str, bb: Backbone, epeg: str) -> Duplex:
    """3' extension (RTT+PBS, 5'->3'), with the epeg motif where it belongs."""
    ext = ext.strip().upper()
    if epeg == "oligo":
        # Oligos carry tevopreQ1; the duplex ends on the vector's TTTT junction.
        return Duplex("extension", "gtgc" + ext + TEVOPREQ1,
                      "aaaa" + rc(TEVOPREQ1) + rc(ext))
    # epeg 'backbone' -> CGCG junction; epeg 'none' -> TTTT junction.
    return Duplex("extension", "gtgc" + ext, rc(bb.down_overhang).lower() + rc(ext))

In [ ]:
# --- QC ----------------------------------------------------------------------

def bsmbi_sites(seq: str) -> list[tuple[int, str]]:
    s = seq.upper()
    hits = [(m.start(), "CGTCTC") for m in _BSMBI_FWD.finditer(s)]
    hits += [(m.start(), "GAGACG") for m in _BSMBI_REV.finditer(s)]
    return sorted(hits)


def qc_design(spacer: str, ext: str, insert_top: str) -> list[str]:
    """Design-level problems that would silently ruin the cloning or the pegRNA."""
    issues: list[str] = []
    for site, where in ((spacer, "protospacer"), (ext, "3' extension")):
        for pos, motif in bsmbi_sites(site):
            issues.append(f"BsmBI site {motif} at {where} position {pos + 1} — assembly will fail")
    # Sites can also be created across a junction, so scan the whole insert.
    extra = len(bsmbi_sites(insert_top))
    if extra:
        issues.append(f"{extra} BsmBI site(s) in the assembled insert (check junctions)")
    for m in _POLYT.finditer(ext.upper()):
        issues.append(
            f"poly-T run ({m.end() - m.start()} nt) at 3' extension position {m.start() + 1} "
            "— may terminate Pol III transcription early"
        )
    return issues


# --- IDT formatting ----------------------------------------------------------

def idt_scale(length: int) -> str:
    """Smallest IDT tube scale that covers this length (15-60/10-90/5-100 nt)."""
    if length <= 60:
        return "25nm"
    if length <= 90:
        return "100nm"
    if length <= 100:
        return "250nm"
    return "ULTRAMER"


# --- oligo naming ------------------------------------------------------------

IDT_NAME_LIMIT = 30


def name_token(text) -> str:
    """Turn a gene symbol or HGVS protein change into an oligo-name token.

    IDT only accepts letters, digits, '_' and '-', so 'p.R451*' becomes 'R451X'
    and 'D594_F595insNS' becomes 'D594F595insNS'. Nonsense '*' maps to 'X' rather
    than being dropped, which would silently make R451* read as R451.
    """
    s = str(text).strip()
    if not s or s.lower() in {"nan", "none"}:
        return ""
    if s[:2].lower() in {"p.", "c.", "g."}:
        s = s[2:]
    return re.sub(r"[^A-Za-z0-9]+", "", s.replace("*", "X"))


def build_oligo_name(prefix: str, index: int, gene: str, mut: str,
                     code: str, tag: str, limit: int = IDT_NAME_LIMIT) -> str:
    """prefix_index_gene_mut_CODE_TAG, trimmed to IDT's name limit.

    The index alone already makes the name unique, so when gene+mut push past the
    limit the labels are shortened (mutation first, then gene) instead of raising.
    """
    head, suffix = f"{prefix}_{index}", f"_{code}_{tag}"
    gene, mut = name_token(gene), name_token(mut)
    budget = limit - len(head) - len(suffix)
    if budget < 0:
        raise ValueError(
            f"{head}{suffix} is already {len(head) + len(suffix)} characters; "
            f"IDT allows {limit}. Shorten `name`."
        )
    for _ in range(2):
        label = "_".join(t for t in (gene, mut) if t)
        over = (len(label) + 1 if label else 0) - budget
        if over <= 0:
            break
        if len(mut) > over:          # trim the mutation, keep the gene whole
            mut = mut[:-over]
        elif mut:
            mut = ""
        else:
            gene = gene[:max(0, len(gene) - over)]
    label = "_".join(t for t in (gene, mut) if t)
    return f"{head}_{label}{suffix}" if label else f"{head}{suffix}"


def well_positions(n: int, order: str = "column") -> list[str]:
    """96-well labels, column-major by default (matches IDT plate upload)."""
    rows, cols = "ABCDEFGH", range(1, 13)
    if order == "column":
        wells = [f"{r}{c}" for c in cols for r in rows]
    else:
        wells = [f"{r}{c}" for r in rows for c in cols]
    if n > len(wells):
        raise ValueError(f"{n} oligos exceed one 96-well plate; split the order.")
    return wells[:n]

In [ ]:
# --- main entry point --------------------------------------------------------

def build_oligos(
    designs: pd.DataFrame,
    name: str,
    backbone: str | Backbone,
    epeg: str | None = None,
    *,
    columns: str | ColumnMap = "pegg",
    gene_col: str | None = "__preset__",
    mut_col: str | None = "__preset__",
    include_scaffold: bool = False,
    g_start: str = "prepend",
    dedup: bool = True,
    plate: bool = False,
    strict: bool = True,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Build IDT-ready oligos for a set of pegRNA designs.

    Returns (orders, report): `orders` is the IDT upload table, `report` is one
    row per design with the oligo names, assembled insert and QC findings.
    """
    bb = resolve_backbone(backbone)
    epeg = resolve_epeg(bb, epeg)
    cols = COLUMN_PRESETS[columns] if isinstance(columns, str) else columns
    if gene_col != "__preset__":
        cols = replace(cols, gene=gene_col)
    if mut_col != "__preset__":
        cols = replace(cols, mut=mut_col)

    for col in (cols.protospacer, cols.extension):
        if col not in designs.columns:
            raise KeyError(f"Column {col!r} not in input ({list(designs.columns)[:8]}...)")
    for col, what in ((cols.gene, "gene_col"), (cols.mut, "mut_col")):
        if col is not None and col not in designs.columns:
            raise KeyError(
                f"{what}={col!r} is not a column of the input "
                f"({list(designs.columns)[:8]}...). Set {what}=None to leave it "
                "out of the oligo names."
            )

    scaffold = scaffold_duplex()
    scaffold.validate()

    order_rows: list[dict] = []
    report_rows: list[dict] = []
    seen: dict[str, str] = {}  # sequence -> first oligo name, for dedup

    proto_series = designs[cols.protospacer].to_numpy()
    ext_series = designs[cols.extension].to_numpy()
    ids = (
        designs[cols.design_id].to_numpy()
        if cols.design_id in designs.columns
        else range(1, len(designs) + 1)
    )
    blanks = [""] * len(designs)
    genes = designs[cols.gene].to_numpy() if cols.gene else blanks
    muts = designs[cols.mut].to_numpy() if cols.mut else blanks

    for n, (raw_proto, raw_ext, design_id, gene, mut) in enumerate(
        zip(proto_series, ext_series, ids, genes, muts), 1
    ):
        proto_in = str(raw_proto)[cols.strip_prefix:]
        ext_in = str(raw_ext)[cols.strip_prefix:]
        spacer, g_note = normalize_protospacer(proto_in, g_start)
        ext = ext_in.strip().upper()

        p = protospacer_duplex(spacer)
        e = extension_duplex(ext, bb, epeg)
        duplexes = [p, scaffold, e] if include_scaffold else [p, e]
        for d in (p, scaffold, e):
            d.validate()

        insert_top = (p.top + scaffold.top + e.top).upper()
        check_assembly_overhangs(bb, [p, scaffold, e], strict=strict)

        issues = qc_design(spacer, ext, insert_top)
        if issues and strict:
            raise ValueError(f"{name}_{n} ({design_id}) failed QC:\n  - " + "\n  - ".join(issues))

        stem = build_oligo_name(name, n, gene, mut, "P", "T")[:-4]

        # entries: (oligo name, sequence, name of the identical oligo already ordered)
        entries: list[tuple[str, str, str | None]] = []
        for d in duplexes:
            code = {"protospacer": "P", "scaffold": "S", "extension": "E"}[d.role]
            for tag, seq in (("T", d.top), ("B", d.bot)):
                oligo_name = build_oligo_name(name, n, gene, mut, code, tag)
                key = seq.upper()
                duplicate_of = seen.get(key) if dedup else None
                if duplicate_of is None:
                    seen[key] = oligo_name
                    order_rows.append(
                        {
                            "Name": oligo_name,
                            "Sequence": seq,
                            "Scale": idt_scale(len(seq)),
                            "Purification": "STD",
                            "Length": len(seq),
                        }
                    )
                entries.append((oligo_name, seq, duplicate_of))

        report_rows.append(
            {
                "oligo_index": n,
                "design_id": design_id,
                "name_stem": stem,
                "gene": name_token(gene),
                "mut": name_token(mut),
                "spacer": spacer,
                "spacer_note": g_note,
                "extension": ext,
                "backbone": bb.name,
                "epeg": epeg,
                "oligos_ordered": ",".join(nm for nm, _, dup in entries if dup is None),
                "reused": ",".join(f"{nm}->{dup}" for nm, _, dup in entries if dup),
                "insert_bp": len(insert_top),
                "qc": "; ".join(issues) if issues else "ok",
            }
        )

    orders = pd.DataFrame(order_rows)
    if plate and not orders.empty:
        orders.insert(0, "Well Position", well_positions(len(orders)))
    return orders, pd.DataFrame(report_rows)


def check_assembly_overhangs(bb: Backbone, duplexes: Sequence[Duplex], strict: bool = True) -> None:
    """Every junction must join a complementary pair of 4-nt overhangs."""
    problems = []
    if duplexes[0].left.upper() != bb.up_overhang:
        problems.append(f"vector {bb.up_overhang} vs protospacer top {duplexes[0].left.upper()}")
    for a, b in zip(duplexes, duplexes[1:]):
        if rc(a.right.upper()) != b.left.upper():
            problems.append(f"{a.role}->{b.role}: {a.right.upper()} / {b.left.upper()}")
    last = duplexes[-1].right.upper()
    if rc(last) != bb.down_overhang:
        problems.append(f"{duplexes[-1].role}->vector: {last} vs {bb.down_overhang}")
    if problems and strict:
        raise ValueError("Overhang mismatch:\n  - " + "\n  - ".join(problems))

In [44]:
# --- in-silico Golden Gate against a real map --------------------------------

def read_snapgene(path: str) -> str:
    from Bio import SeqIO

    return str(SeqIO.read(path, "snapgene").seq).upper()


def digest_bsmbi(seq: str) -> tuple[str, str, str]:
    """Digest a circular plasmid with exactly two BsmBI sites.

    Returns (backbone_top, up_overhang, down_overhang) where backbone_top starts
    with the downstream 5' top overhang and ends flush before the upstream one.
    """
    fwd = [m.start() for m in _BSMBI_FWD.finditer(seq)]
    rev = [m.start() for m in _BSMBI_REV.finditer(seq)]
    if len(fwd) != 1 or len(rev) != 1:
        raise ValueError(f"Expected one site per strand, found {len(fwd)} CGTCTC / {len(rev)} GAGACG")
    i, j = fwd[0], rev[0]
    down = seq[i + 7:i + 11]           # 5' top overhang on the backbone
    up = seq[j - 5:j - 1]              # top-strand spelling of the bottom overhang
    backbone_top = seq[i + 7:] + seq[:j - 5] if j < i else seq[i + 7:j - 5]
    return backbone_top, up, down


def backbone_from_map(path: str, name: str | None = None) -> Backbone:
    """Derive a Backbone from a SnapGene map rather than trusting the registry."""
    seq = read_snapgene(path)
    _, up, down = digest_bsmbi(seq)
    return Backbone(name or path.rsplit("/", 1)[-1], up, down, TEVOPREQ1 in seq)


def assemble(vector_map: str, spacer: str, ext: str, epeg: str) -> dict:
    """Run the Golden Gate in silico and verify the product."""
    seq = read_snapgene(vector_map)
    backbone_top, up, down = digest_bsmbi(seq)
    bb = Backbone(vector_map.rsplit("/", 1)[-1], up, down, TEVOPREQ1 in seq)

    p, s, e = protospacer_duplex(spacer), scaffold_duplex(), extension_duplex(ext, bb, epeg)
    for d in (p, s, e):
        d.validate()
    check_assembly_overhangs(bb, [p, s, e])

    insert_top = (p.top + s.top + e.top).upper()
    product = backbone_top + insert_top
    expected_cassette = (
        OH_VECTOR_UP + spacer + "GTTTTAGAG" + SCAFFOLD + OH_SCAF_EXT + ext
        + (TEVOPREQ1 if epeg == "oligo" else "")
    )
    return {
        "backbone": bb,
        "product_bp": len(product),
        "product": product,
        "insert_bp": len(insert_top),
        "cassette_ok": expected_cassette.upper() in product,
        "residual_bsmbi": bsmbi_sites(product + product[:5]),
        "tevo_in_product": product.count(TEVOPREQ1),
    }

## 1. Load the designs

Either point `INPUT` at a csv, or replace `DESIGNS` with a DataFrame straight from PEGG:

```python
from pegg import prime
pegRNAs = prime.run(input, "PrimeDesign", PAM="NGG", PBS_lengths=[7,9,11,13,15,17], RTT_lengths=[15])
DESIGNS = pegRNAs
```

In [ ]:
import os

DEMO = pd.DataFrame(
    {
        "Protospacer": ["GGCGAAGTTGGCGATGCCGA", "GGGACTCGCAGCGCACGTTC",
                        "GTGGACCCCAGCAATGCCAT", "TGGACCCCAGCAATGCCAT"],
        "RTT_PBS": ["AATGGCATCGGCATCGCCAACTT", "ACGACGTGAACGTGCGCTGCGAGTC",
                    "TTGGCGATGCCGATTGCATTGCTGGGGT", "TTGGCGATTCCGATGGCATTGCTGGGGTCC"],
        "pegRNA_id": ["pegRNA_259", "pegRNA_90", "pegRNA_326", "pegRNA_328"],
        "Hugo_Symbol": ["Keap1", "Keap1", "Keap1", "Keap1"],
        "HGVSp_m": ["G333C", "G333C", "R470C", "R470C"],
    }
)

if os.path.exists(INPUT):
    DESIGNS = pd.read_csv(INPUT)
else:
    DESIGNS = DEMO
    print(f"{INPUT} not found — running on the built-in demo set (KEAP1 guides).")

print(f"{len(DESIGNS)} designs loaded — all of them will be ordered.")
cols = COLUMN_PRESETS[COLUMNS]
DESIGNS[[c for c in (cols.protospacer, cols.extension, cols.design_id,
                     GENE_COL, MUT_COL) if c and c in DESIGNS]]

## 2. Build the oligos

`include_scaffold=False` (the default) keeps the scaffold pair out of the order sheet.

```{tip}
`strict=True` turns any QC finding into an exception. That is what you want before spending
money; pass `strict=False` to see all findings at once while troubleshooting a design.
```

In [ ]:
orders, report = build_oligos(
    DESIGNS,
    name=NAME,
    backbone=BACKBONE,
    epeg=EPEG,
    columns=COLUMNS,
    gene_col=GENE_COL,
    mut_col=MUT_COL,
    include_scaffold=False,   # scaffold comes from phosphorylated stock
    g_start="prepend",        # U6 needs a G at +1; prepend keeps the spacer intact
    dedup=True,
    plate=PLATE,
    strict=True,
)

print(f"{len(orders)} oligos to order "
      f"({len(DESIGNS)} pegRNAs x 4, minus {len(DESIGNS) * 4 - len(orders)} shared)")
orders

### Per-design report

In [ ]:
report[["oligo_index", "name_stem", "design_id", "spacer", "spacer_note",
        "epeg", "insert_bp", "reused", "qc"]]

## 3. In-silico Golden Gate

Set `VECTOR_MAP` to the SnapGene `.dna` file. This digests the plasmid with BsmBI, ligates
the annealed duplexes by overhang, and checks the product: the cassette reads
`U6 … CACC-protospacer-scaffold-RTT-PBS[-tevopreQ1]-TTTT`, with zero BsmBI sites left.

```{warning}
`backbone_from_map` reads the overhangs out of the map instead of trusting the registry. If it
disagrees with `BACKBONE`, believe the map — you may be holding a different vector.
```

In [ ]:
if VECTOR_MAP:
    derived = backbone_from_map(VECTOR_MAP, os.path.basename(VECTOR_MAP))
    print(f"map:      {derived.name}")
    print(f"          overhangs {derived.up_overhang} / {derived.down_overhang}, "
          f"tevopreQ1 in backbone: {derived.has_tevopreQ1}")
    print(f"          valid epeg options: {derived.epeg_options()}")

    registry = resolve_backbone(BACKBONE)
    if (derived.up_overhang, derived.down_overhang, derived.has_tevopreQ1) != (
        registry.up_overhang, registry.down_overhang, registry.has_tevopreQ1
    ):
        raise ValueError(
            f"VECTOR_MAP does not match BACKBONE={BACKBONE!r}: the map has overhangs "
            f"{derived.up_overhang}/{derived.down_overhang} and tevopreQ1="
            f"{derived.has_tevopreQ1}. Trust the map — the oligos above are for the wrong vector."
        )

    epeg = resolve_epeg(derived, EPEG)
    for row in report.itertuples():
        r = assemble(VECTOR_MAP, row.spacer, row.extension, epeg)
        flag = "OK " if (r["cassette_ok"] and not r["residual_bsmbi"]) else "FAIL"
        print(f"  {flag} {row.name_stem}: {r['product_bp']} bp, "
              f"cassette_ok={r['cassette_ok']}, "
              f"residual BsmBI={len(r['residual_bsmbi'])}, "
              f"tevopreQ1 copies={r['tevo_in_product']}")
else:
    print("VECTOR_MAP not set — skipping in-silico assembly.\n"
          "Overhang compatibility was still enforced during build_oligos().")

## 4. Export for IDT

Upload `*_oligos.csv` to IDT → Custom DNA Oligos → **Bulk input / Upload**. Tube orders take
Name/Sequence/Scale/Purification; set `PLATE=True` for the plate template's Well Position.

In [ ]:
base = os.path.join(OUTPUT_DIR, f"{NAME}_{BACKBONE}")
orders.drop(columns=["Length"]).to_csv(f"{base}_oligos.csv", index=False)
report.to_csv(f"{base}_report.csv", index=False)
DESIGNS.to_csv(f"{base}_designs.csv", index=False)

print(f"wrote {base}_oligos.csv   ({len(orders)} oligos)")
print(f"      {base}_report.csv")
print(f"      {base}_designs.csv")
print("\nscale breakdown:")
print(orders.groupby("Scale").size().to_string())

## Scaffold oligos (stock reference)

Not in the order sheet — phosphorylate and anneal these once, keep at −20 °C, reuse. Order
again only when stock runs out. Both are 67 nt → **100 nm** scale.

In [ ]:
s = scaffold_duplex()
s.validate()
pd.DataFrame(
    [
        {"Name": "pegRNA_scaffold_T", "Sequence": s.top,
         "Scale": idt_scale(len(s.top)), "Purification": "STD"},
        {"Name": "pegRNA_scaffold_B", "Sequence": s.bot,
         "Scale": idt_scale(len(s.bot)), "Purification": "STD"},
    ]
)

## Wet-lab hand-off

1. Resuspend oligos to **100 µM**.
2. Anneal protospacer and 3' extension pairs with **T4 DNA Ligase** (no PNK); scaffold stock is
   annealed with **T4 PNK**. Program: 37 °C 30 min → 95 °C 5 min → ramp to 25 °C at 5 °C/min.
3. Dilute annealed products **1:100** in NF water.
4. Golden Gate: 1.5 µL vector (75 ng/µL) + 1 µL each of protospacer / scaffold / 3' ext (100 nM)
   + 2 µL 10× T4 ligase buffer + 1 µL NEB Golden Gate Mix (BsmBI) + 12.5 µL H₂O.
   `(42 °C 1 min → 16 °C 1 min) × 60 → 60 °C 5 min`. Transform the whole 20 µL.

```{note}
The `CGCG` junction used by the `-tevo` backbones is palindromic, so the 3' extension duplex can
ligate to itself. If you see a high background of small colonies on a `-tevo` cloning, that is
the likely cause; the `TTTT`/`AAAA` junction of plain UPEmS/UPEC does not have this issue.
```

## Appendix: regression test

Rebuilds oligos for rows taken verbatim from the lab's own spreadsheets and asserts they match.
Run this after editing anything above — a silent change here costs an oligo order.

In [ ]:
# (protospacer, extension, Ptop, Pbot, Etop, Ebot) straight from the spreadsheets
GROUND_TRUTH = {
    # Into UPEmS-tevo / UPEC-tevo  (sheet 17y-Qk6u..., KEAP1 set)
    ("UPEmS-tevo", "backbone"): [
        ("GGCGAAGTTGGCGATGCCGA", "AATGGCATCGGCATCGCCAACTT",
         "caccGGCGAAGTTGGCGATGCCGAgtttt", "ctctaaaacTCGGCATCGCCAACTTCGCC",
         "gtgcAATGGCATCGGCATCGCCAACTT", "cgcgAAGTTGGCGATGCCGATGCCATT"),
        ("GTGTCGGAAGTAGCCGCCCG", "CGCGTGCGGCTACTTCCG",
         "caccGTGTCGGAAGTAGCCGCCCGgtttt", "ctctaaaacCGGGCGGCTACTTCCGACAC",
         "gtgcCGCGTGCGGCTACTTCCG", "cgcgCGGAAGTAGCCGCACGCG"),
    ],
    # Into UPEmS / UPEC without epeg motif  (sheet 1M6cFbNP..., Tet2_R1444* set)
    ("UPEmS", "none"): [
        ("GCGTGGAACTGCTCATCCTC",
         "AATAGAGAAGACAATtGAGAAGTCGGAGCTAAGCCTGAGGATGAGCAGTT",
         "caccGCGTGGAACTGCTCATCCTCgtttt", "ctctaaaacGAGGATGAGCAGTTCCACGC",
         "gtgcAATAGAGAAGACAATtGAGAAGTCGGAGCTAAGCCTGAGGATGAGCAGTT",
         "aaaaAACTGCTCATCCTCAGGCTTAGCTCCGACTTCTCAATTGTCTTCTCTATT"),
    ],
}

problems = []
for (bb_name, epeg_mode), rows in GROUND_TRUTH.items():
    bb = resolve_backbone(bb_name)
    for proto, ext, ptop, pbot, etop, ebot in rows:
        p, e = protospacer_duplex(proto), extension_duplex(ext, bb, epeg_mode)
        for label, got, want in (("P_T", p.top, ptop), ("P_B", p.bot, pbot),
                                 ("E_T", e.top, etop), ("E_B", e.bot, ebot)):
            if got.upper() != want.upper():
                problems.append(f"{bb_name}/{epeg_mode} {proto[:8]} {label}: {got} != {want}")

# epeg='oligo' has no spreadsheet; check it against the protocol document's diagram
e = extension_duplex("AATGGCATCGGCATCGCCAACTT", resolve_backbone("UPEmS"), "oligo")
if e.top != "gtgcAATGGCATCGGCATCGCCAACTT" + TEVOPREQ1:
    problems.append("epeg='oligo' top oligo")
if e.bot != "aaaa" + rc(TEVOPREQ1) + "AAGTTGGCGATGCCGATGCCATT":
    problems.append("epeg='oligo' bottom oligo")

s = scaffold_duplex()
if s.top != "agag" + SCAFFOLD or s.bot != "gcac" + rc(SCAFFOLD):
    problems.append("scaffold oligos")

# Oligo names: NAME_index_gene_mut_CODE_TAG, sanitized and within IDT's 30 chars
NAME_CASES = [
    (("CH_AML", 37, "Tet2", "G1105*", "P", "T"), "CH_AML_37_Tet2_G1105X_P_T"),
    (("CH_AML", 1, "Asxl1", "p.G643Wfs*14", "E", "B"), "CH_AML_1_Asxl1_G643WfsX14_E_B"),
    (("CH_AML", 63, None, None, "P", "B"), "CH_AML_63_P_B"),
]
for args, want in NAME_CASES:
    got = build_oligo_name(*args)
    if got != want:
        problems.append(f"name {args}: {got} != {want}")

# Long labels are trimmed, not rejected, and stay unique through the index
long_name = build_oligo_name("CH_AML", 8, "Flt3", "D594_F595insNS", "P", "T")
if len(long_name) > IDT_NAME_LIMIT or not long_name.startswith("CH_AML_8_Flt3_D594"):
    problems.append(f"long label not trimmed sensibly: {long_name}")
if build_oligo_name("CH_AML", 8, "Flt3", "D594_F595insNS", "P", "B") == long_name:
    problems.append("top/bottom oligos got the same name")

# Impossible backbone/epeg combinations must raise, not produce dead oligos
for bad_bb, bad_epeg in (("UPEmS", "backbone"), ("UPEmS-tevo", "oligo"),
                         ("UPEmS-tevo", "none")):
    try:
        resolve_epeg(resolve_backbone(bad_bb), bad_epeg)
        problems.append(f"{bad_bb} + epeg={bad_epeg!r} was accepted")
    except ValueError:
        pass

print("\n".join(problems) if problems else
      "all ground-truth oligos reproduced; invalid backbone/epeg combinations rejected")